# Gaming_Data_Analysis
Analysis of consumer trends in video game data

The main question this analysis aims to answer is what correlation if any exists between a video game's sales and any negative impact on the mental or social well-being of its players.  Video games are an increasingly profitable sector of the entertainment industry, eclipsing both the music and movie industries.  However, concerns about their possible impact upon the psychosocial health of its players have been raised from multiple corners.  In this analysis several data sets will be examined, with visualizations provided on a Tableau dashboard.

## Project Objective 

The objective of this project is to see what correlations can be found between a video game's sales and several other factors.  These include several mental health screening evaluations as well as the genre.  Questions to answer include whether game genre shows any noticeable correlation to mental health scores, genre profitability, and any possible correlation between the gamer's mental and social health and overall game profits.


Our first step is to import our needed libraries and our datasets.

In [115]:
#Import libraries

import pandas as pd
import sqlite3

#Read each csv and create a corresponding pandas dataframe.

sales_data = pd.read_csv("Video_Game_Sales_1978-2024.csv")
top_50_data = pd.read_csv("2024_Top_50_AAA_AA_Indie_Games.csv")
gaming_study_data = pd.read_csv("GamingStudy_data.csv", encoding='windows-1253')
online_behavior_data = pd.read_csv("online_gaming_behavior_dataset.csv")
genres = pd.read_csv('genres.csv')



Let's start by examing our initial data.

In [116]:


print(sales_data.head(), '\n')
print(sales_data.info(), '\n')

print(top_50_data.head(), '\n')
print(top_50_data.info(), '\n')

print(gaming_study_data.head(), '\n')
print(gaming_study_data.info(), '\n')

print(online_behavior_data.head(), '\n')
print(online_behavior_data.info(), '\n')

print(genres.head(), '\n')
print(genres.info())

   Rank              Name Platform All_Platforms  \
0     1            Tetris   Series           NaN   
1     2           Pokemon   Series           NaN   
2     3      Call of Duty   Series           NaN   
3     4  Grand Theft Auto   Series           NaN   
4     5       Super Mario   Series           NaN   

                                           All_Games             Publisher  \
0  Tetris (1984)|Tetris (1989)|Welltris|Hatris|Tw...  The Tetris Company     
1  Pokemon Red & Green (Japan-only) & Blue|Pokemo...            Nintendo     
2  Call of Duty|Call of Duty 2|Call of Duty 3|Cal...          Activision     
3  Grand Theft Auto|Grand Theft Auto: London 1969...      Rockstar Games     
4  Mario Bros.|Super Mario Bros.|Super Mario Bros...            Nintendo     

           Developer  Critic_Score  User_Score  NA_Sales  PAL_Sales  JP_Sales  \
0  Alexey Pajitnov             NaN         NaN       NaN        NaN       NaN   
1       Game Freak             NaN         NaN       NaN

Now that we have an idea of our data's shape and contents, it is time to start cleaning and normalizing our datasets.

In [117]:
#Add normalization of genre data to top_50_data
top_50_data = pd.merge(top_50_data, genres[['Name', 'Genre']], on='Name', how='left')

#Align column names and rename columns for clarity.

top_50_data.rename(columns={"ReleaseDate": "Year"}, inplace=True)
gaming_study_data.rename(columns={"Game": "Name", "Hours": "HoursPerWeek", "Residence_ISO3": "Location", "GAD_T": "GAD_Total", "SWL_T": "SWL_Total", "SPIN_T": "SPIN_Total"}, inplace=True)
online_behavior_data.rename(columns={"GameGenre": "Genre", "PlayTimeHours": "HoursPerWeek", }, inplace=True)



In [118]:
#Drop unnecessary columns from dataframe.

sales_data = sales_data.drop(['Rank', 'Publisher', 'Developer', 'Critic_Score', 'All_Platforms', 'User_Score', 'All_Games', 'NA_Sales', 'PAL_Sales', 'JP_Sales', 'Other_Sales'], axis=1)
top_50_data = top_50_data.drop(['Publishers', 'Developers', 'Steam Id', 'Price', 'Review Count', 'Review Score', 'Steam Followers'], axis=1)
top_50_data = top_50_data.drop(top_50_data.columns[0], axis=1)
gaming_study_data = gaming_study_data.drop(['S. No.', 'Timestamp', 'GADE', 'earnings', 'whyplay', 'Degree', 'Playstyle', 'Work', 'League', 'highestleague', 'streams', 'Birthplace', 'Residence', 'Reference', 'accept', 'Birthplace_ISO3'], axis=1)
online_behavior_data = online_behavior_data.drop(['PlayerID', 'PlayerLevel', 'AchievementsUnlocked'], axis=1)

print(sales_data.head())
print(top_50_data.head())
print(gaming_study_data.head())
print(online_behavior_data.head())

               Name Platform  Global_Sales    Year             Genre
0            Tetris   Series           NaN  1988.0            Puzzle
1           Pokemon   Series           NaN  1998.0      Role-Playing
2      Call of Duty   Series           NaN  2003.0           Shooter
3  Grand Theft Auto   Series           NaN  1998.0  Action-Adventure
4       Super Mario   Series           NaN  1983.0          Platform
                               Name       Year    Copies Sold  \
0                Black Myth: Wukong  8/19/2024  21,836,817.00   
1                      HELLDIVERS 2   2/8/2024  12,410,685.00   
2                          Palworld  1/18/2024  18,567,241.00   
3  Warhammer 40,000: Space Marine 2   9/9/2024   2,709,725.00   
4                   Path of Exile 2  12/6/2024   5,015,358.00   

     Gross Revenue  Average Playtime (Hrs) PublisherClass  Early Access  \
0  $951,505,160.00                    60.4            AAA         False   
1  $443,622,746.00                    63.4   

In [119]:
#Remove rows with NA for NAME
top_50_data = top_50_data.dropna(subset='Name')
sales_data = sales_data.dropna(subset='Name')
gaming_study_data = gaming_study_data.dropna(subset='Name')

We want to make sure all of our dataset columns are the expected datatype.  

In [120]:
#Convert any necessary columns to correct data type
sales_data['Name'] = sales_data['Name'].astype("string")
sales_data['Platform'] = sales_data['Platform'].astype("string")
sales_data['Genre'] = sales_data['Genre'].astype("string")

print(sales_data.info())

top_50_data['Name'] = top_50_data['Name'].astype("string")
top_50_data['Genre'] = top_50_data['Genre'].astype("string")
top_50_data['PublisherClass'] = top_50_data['PublisherClass'].astype("string")
top_50_data['Copies Sold'] = top_50_data['Copies Sold'].str.replace(',', '')
top_50_data['Copies Sold'] = top_50_data['Copies Sold'].astype('float64')
top_50_data['Copies Sold'] = top_50_data['Copies Sold'].astype(int)
top_50_data['Gross Revenue'] = top_50_data['Gross Revenue'].str.replace(',', '')
top_50_data['Gross Revenue'] = top_50_data['Gross Revenue'].str.replace('$', '')
top_50_data['Gross Revenue'] = top_50_data['Gross Revenue'].astype('float64')


print(top_50_data.info())

gaming_study_data['Name'] = gaming_study_data['Name'].astype("string")
gaming_study_data['Gender'] = gaming_study_data['Gender'].astype("string")
gaming_study_data['Platform'] = gaming_study_data['Platform'].astype("string")
gaming_study_data['Location'] = gaming_study_data['Location'].astype("string")

print(gaming_study_data.info())

online_behavior_data['Gender'] = online_behavior_data['Gender'].astype("string")
online_behavior_data['Location'] = online_behavior_data['Location'].astype("string")
online_behavior_data['Genre'] = online_behavior_data['Genre'].astype("string")
online_behavior_data['GameDifficulty'] = online_behavior_data['GameDifficulty'].astype("string")
online_behavior_data['EngagementLevel'] = online_behavior_data['EngagementLevel'].astype("string")

print(online_behavior_data.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63927 entries, 0 to 63926
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Name          63927 non-null  string 
 1   Platform      63927 non-null  string 
 2   Global_Sales  20301 non-null  float64
 3   Year          57062 non-null  float64
 4   Genre         63927 non-null  string 
dtypes: float64(2), string(3)
memory usage: 2.4 MB
None
<class 'pandas.core.frame.DataFrame'>
Index: 149 entries, 0 to 149
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Name                    149 non-null    string 
 1   Year                    149 non-null    object 
 2   Copies Sold             149 non-null    int64  
 3   Gross Revenue           149 non-null    float64
 4   Average Playtime (Hrs)  149 non-null    float64
 5   PublisherClass          149 non-null    string 
 6   Early A

More data cleaning...

In [121]:
#Remove rows with NA for SPIN
gaming_study_data = gaming_study_data.dropna(subset='SPIN_Total')

#Clean up the Platform data - only PC and console are differentiated so no need for the special characters. Similarly Smartphone / Tablet can be consolidated as Mobile.
gaming_study_data['Platform'] = gaming_study_data['Platform'].replace('Console (PS, Xbox, ...)', 'Console')
gaming_study_data['Platform'] = gaming_study_data['Platform'].replace('Smartphone / Tablet', 'Mobile')

#The names of the games on the gaming study data should match those on the sales data sheet as well.

gaming_study_data['Name'] = gaming_study_data['Name'].replace('Skyrim', 'The Elder Scrolls V: Skyrim')
gaming_study_data['Name'] = gaming_study_data['Name'].replace('Counter Strike', 'Counter-Strike')
gaming_study_data['Name'] = gaming_study_data['Name'].replace('Diablo 3', 'Diablo III')

#Clean up the Platform data - only PC and console are differentiated so no need for the special characters. Similarly Smartphone / Tablet can be consolidated as Mobile.
gaming_study_data['Platform'] = gaming_study_data['Platform'].replace('Console (PS, Xbox, ...)', 'Console')
gaming_study_data['Platform'] = gaming_study_data['Platform'].replace('Smartphone / Tablet', 'Mobile')

#The names of the games on the gaming study data should match those on the sales data sheet as well.

gaming_study_data['Name'] = gaming_study_data['Name'].replace('Skyrim', 'The Elder Scrolls V: Skyrim')
gaming_study_data['Name'] = gaming_study_data['Name'].replace('Counter Strike', 'Counter-Strike')
gaming_study_data['Name'] = gaming_study_data['Name'].replace('Diablo 3', 'Diablo III')

#Remove all game series from sales data as well as cross-platform entries as these have no sales

sales_data = sales_data.drop(sales_data[(sales_data['Platform'] == ('Series'))].index)
sales_data = sales_data.dropna(subset="Global_Sales")

#Update PokÃ©mon to Pokemon in the sales data. As there are various versions of 'Starcraft II' on the sales data sheet this is updated to 'Starcraft' to match the gaming study data.

sales_data['Name'] = sales_data['Name'].str.replace('PokÃ©mon', 'Pokemon')
sales_data.loc[sales_data['Name'].str.contains('StarCraft II'), 'Name'] = 'Starcraft 2'

online_behavior_data['Genre'] = online_behavior_data['Genre'].replace('RPG', 'Role-Playing')



print(gaming_study_data.head())

   GAD1  GAD2  GAD3  GAD4  GAD5  GAD6  GAD7  SWL1  SWL2  SWL3  ...  SPIN15  \
0     0     0     0     0     1     0     0     3     5     5  ...     0.0   
1     1     2     2     2     0     1     0     3     5     2  ...     3.0   
2     0     2     2     0     0     3     1     2     6     5  ...     4.0   
3     0     0     0     0     0     0     0     2     5     5  ...     1.0   
4     2     1     2     2     2     3     2     2     2     4  ...     0.0   

   SPIN16 SPIN17 Narcissism  Gender  Age  GAD_Total  SWL_Total  SPIN_Total  \
0     1.0    0.0        1.0    Male   25          1         23         5.0   
1     1.0    2.0        1.0    Male   41          8         16        33.0   
2     4.0    2.0        4.0  Female   32          8         17        31.0   
3     0.0    0.0        2.0    Male   28          0         17        11.0   
4     3.0    0.0        1.0    Male   19         14         14        13.0   

   Location  
0       USA  
1       USA  
2       DEU  
3     

It is time to start writing our utility functions.  Because many titles in the Video Game Sales dataset are duplicated, we need to ensure that each name is unique in our data set before we can load it into SQL.  Additionally, we want to be careful about which titles we include - Starcraft and Starcraft 2 are not the same video game.

In [122]:
#Utility function to get average value of one specified column based on given value in another column (defaults to Name).

def get_average(temp_df, tlist: list[str], cname: str, kname: str='Name'):
    """Calculates average value of specified column based on given value in another column (defaults to Name).
    Inputs:  Pandas dataframe, list of unique values for key column, name of column to calculate, optional key column specification.
    Outputs:  No return value; appends "cname_avg" column to temp_df."""
    for i in range(len(tlist)):
        mask = temp_df[kname] == tlist[i]
        average_val = temp_df[temp_df[kname] == tlist[i]][cname].mean()
        temp_df.loc[temp_df[kname] == tlist[i], [cname+'_avg']]= average_val

In [123]:
def duplist(temp_df):
    """Provides a list of values in the Name field  of the sales_data that appear more than once.  Only exact matches are returned.
    For example, 'Tetris', 'Tetris DS', and 'Tetris Plus' are all considered unique values."""
    multi_series = temp_df['Name'].value_counts()
    multi_list = multi_series[multi_series > 1].index.to_list()
    multi_list = list(set(multi_list))
    return multi_list

In [124]:
#If there is a title with multiple rows but no row where the Platform value is All, create a new row for the title.
#Genre and Year values will be based taken from the earliest release year.

mask = sales_data['Name'].isin(duplist(sales_data))
placeholder = sales_data[mask] #dataframe with only titles with more than one entry
placeholder = placeholder[placeholder['Global_Sales'] != 0.0]
plist = placeholder['Name'].to_list()

title_list = [] #A list of dataframes, with each list element being for one game title

for i in range(len(plist)):
    temp_df = placeholder[placeholder['Name'] == plist[i]]
    title_list.append(temp_df)

In [125]:
def title_gs_calc(title_df):
    """Takes a dataframe with multiple rows for a title and returns a dataframe with a single row with the Global_Sales calculated."""
    return_df = title_df.head(1).reset_index()
    return_df['Platform'] = 'All'
    return_df['Global_Sales'] = title_df['Global_Sales'].sum()    
    return return_df

In [126]:
def title_add(tlist):
    """Takes a list of dataframes consisting of the same title and returns a dataframe with All for the Platform value and the sum of the columns for Global_Sales."""
    added_titles = pd.DataFrame()
    for i in range(len(tlist)):
        temp_df = title_gs_calc(tlist[i])
        added_titles = pd.concat([added_titles, temp_df], ignore_index=True)
        added_titles = added_titles.drop_duplicates()     
    return added_titles

Now that we have written our utility functions, it is time to use them to process our sales data.

In [127]:
sales_data = pd.concat([sales_data, title_add(title_list)], ignore_index=True)
sales_data = sales_data.reset_index(drop=True)

remove_title_list = duplist(sales_data)
rows_to_remove = sales_data[sales_data['Name'].isin(remove_title_list)].index.to_list()
rows_to_save = sales_data[sales_data['Platform'] == 'All']

sales_data = sales_data.drop(rows_to_remove)
sales_data = pd.concat([sales_data, rows_to_save], ignore_index=True)
sales_data = sales_data.drop_duplicates()
sales_data = sales_data.drop(columns=sales_data.columns[5], axis=1)
print(sales_data.tail()) #As the new rows should be added to the bottom of the dataframe, we want to check the tail rather than the head.

                    Name Platform  Global_Sales    Year      Genre
13966  Iwaihime: Matsuri      All          0.01  2017.0  Adventure
13967   Closed Nightmare      All          0.01  2018.0       Misc
13968          Teslagrad      All          0.01  2014.0   Platform
13969             Island      All          0.01  2017.0  Adventure
13970     Super Meat Boy      All          0.01  2016.0   Platform


Pulling singleplayer and multiplayer value from the Tags column.

In [128]:
#Create list from Name column of top_50_data to use for column name for new dataframe created from Tags column
top_50_names = top_50_data['Name'].tolist()
top_50_data.index = top_50_names
top_50_data['Singleplayer'] = False
top_50_data['Multiplayer'] = False

In [129]:
#Utility function to check whether an element exists in a list of strings.

def list_check(selement: str, lelement: list[str]):
    """For a dataframe column consisting of lists of strings, return True if the tag exists in the list and False if it does not."""
    return selement in lelement

#Find Singleplayer and Multiplayer tags and set value of corresponding column to True if found

for name in top_50_names:
    taglist = top_50_data.loc[name, 'Tags']
    if list_check('Singleplayer', taglist):
        top_50_data.loc[name, 'Singleplayer'] = True
    else:
        top_50_data.loc[name, 'Singleplayer'] = False    
    if list_check('Multiplayer', taglist):
        top_50_data.loc[name, 'Multiplayer'] = True
    else:
        top_50_data.loc[name, 'Multiplayer'] = False

#Now that we have the Singleplayer and Multiplayer tags extracted we can drop the Tags column
top_50_data = top_50_data.drop(columns='Tags')

print(top_50_data.head())

                                                              Name       Year  \
Black Myth: Wukong                              Black Myth: Wukong  8/19/2024   
HELLDIVERS 2                                          HELLDIVERS 2   2/8/2024   
Palworld                                                  Palworld  1/18/2024   
Warhammer 40,000: Space Marine 2  Warhammer 40,000: Space Marine 2   9/9/2024   
Path of Exile 2                                    Path of Exile 2  12/6/2024   

                                  Copies Sold  Gross Revenue  \
Black Myth: Wukong                   21836817    951505160.0   
HELLDIVERS 2                         12410685    443622746.0   
Palworld                             18567241    435465541.0   
Warhammer 40,000: Space Marine 2      2709725    142612930.0   
Path of Exile 2                       5015358    135369527.0   

                                  Average Playtime (Hrs) PublisherClass  \
Black Myth: Wukong                                  6

In [130]:
#Get column averages for specified and append to new columns at end

gmask = gaming_study_data['Name'].unique().tolist()

get_average(gaming_study_data, gmask, 'GAD_Total')
get_average(gaming_study_data, gmask, 'SWL_Total')
get_average(gaming_study_data, gmask, 'SPIN_Total')
get_average(gaming_study_data, gmask, 'HoursPerWeek')

#Create new dataframe with one entry per game.  The individual item averages for each screening test are not included in the dataframe
# as it is intended to be a high-level overview.

gaming_study_avg = gaming_study_data.filter(['Name', 'HoursPerWeek_avg', 'GAD_Total_avg', 'SWL_Total_avg', 'SPIN_Total_avg'], axis=1)
gaming_study_avg = gaming_study_avg.drop_duplicates()

Let's use SQL to join our two main datasets.  All of our SQL code will run in one code cell to ensure the SQL connection is properly closed.  Additionally, while not currently used, a cursor is created to allow for ease of further use if desired.

In [131]:
#Open the SQL connection and create a cursor

conn = sqlite3.connect(':memory:') 
cursor = conn.cursor()

def sql(query):
    """Input a SQL query and return a pandas dataframe object."""
    return pd.read_sql_query(query, conn)

#Load our data into SQL tables

sales_data.to_sql('Sales', conn, if_exists='replace', index=False)
gaming_study_avg.to_sql('Gaming_Study_Avg', conn, if_exists='replace', index=False)

conn.execute("""
    CREATE TABLE SalesT
    (
    Name TEXT PRIMARY KEY,
    Platform TEXT,
    Global_Sales REAL,
    Genre TEXT,
    Year INTEGER);
    """)
    
conn.execute("""
    INSERT OR REPLACE INTO SalesT (Name, Platform, Global_Sales, Genre, Year)
    SELECT Name, Platform, Global_Sales, Genre, Year FROM Sales;
    """)

conn.execute("""
    CREATE TABLE GST
    (
    Name TEXT PRIMARY KEY,
    HoursPerWeek_avg REAL,
    GAD_Total_avg REAL,
    SWL_Total_avg REAL,
    SPIN_Total_avg REAL);
    """)

conn.execute("""
    INSERT OR REPLACE INTO GST (Name, HoursPerWeek_avg, GAD_Total_avg, SWL_Total_avg, SPIN_Total_avg)
    SELECT Name, HoursPerWeek_avg, GAD_Total_avg, SWL_Total_avg, SPIN_Total_avg FROM Gaming_Study_Avg;
    """)

sales_and_study = sql("""
    SELECT *
    FROM GST
    LEFT JOIN SalesT ON SalesT.Name = GST.Name;
    """)
sales_and_study = sales_and_study.loc[:,~sales_and_study.columns.duplicated()]

conn.close()

#Due to Other not being a single video game, and League of Legends and Hearthstone not appearing on the sales data due to being free to download, we must manually assign Platform and Genre.

sales_and_study.iat[1,5] = 'All'
sales_and_study.iat[3,5] = 'PC'
sales_and_study.iat[9,5] = 'All'

sales_and_study.iat[3,7] = 'Strategy'
sales_and_study.iat[9,7] = 'Strategy'

sales_and_study.iat[3,8] = 2009
sales_and_study.iat[9,8] = 2014

#Missed updating these column names earlier.

sales_data.rename(columns={"Global_Sales": "Global Sales"}, inplace=True)
top_50_data.rename(columns={"Gross Revenue": "Global Sales"}, inplace=True)

print(sales_and_study)

                           Name  HoursPerWeek_avg  GAD_Total_avg  \
0   The Elder Scrolls V: Skyrim         23.304348       5.086957   
1                         Other         22.965944       5.461934   
2             World of Warcraft         26.496644       5.127517   
3             League of Legends         22.147442       5.193539   
4                   Starcraft 2         16.667692       4.630769   
5                Counter-Strike         25.439597       5.136667   
6                       Destiny         19.555556       3.388889   
7                    Diablo III         28.903614       5.951807   
8           Heroes of the Storm         20.250000       5.425000   
9                   Hearthstone         16.884211       5.852632   
10                 Guild Wars 2         24.527778       4.333333   

    SWL_Total_avg  SPIN_Total_avg Platform  Global_Sales         Genre    Year  
0       21.782609       23.608696      All         20.51  Role-Playing  2011.0  
1       19.504115    

Almost done!  Our final datasets can be found in the Final_Data folder of the folder the repository was cloned to.  All visualizations can be accessed from the Tableau dashboard at https://public.tableau.com/app/profile/jud.singleton/viz/GamingDataAnalysisCapstone/GamingDataAnalysisCapstone?publish=yes.

In [132]:
#Export our updated and new dataframes to CSV to be loaded into Tableau!

sales_data.to_csv('Final_Data/Sales.csv', index=False)
top_50_data.to_csv('Final_Data/Top_50.csv', index=False)
gaming_study_data.to_csv('Final_Data/Gaming_Study.csv', index=False)
online_behavior_data.to_csv('Final_Data/Behavior.csv', index=False)
sales_and_study.to_csv('Final_Data/Sales_and_Study.csv', index=False)